# 01 — Information capitalization of contracted data revenue

**Study:** *Does the Market Price Contracted Data Revenue? Backlog Depth as an Information Signal in the Valuation of Listed Earth-Observation Firms.*

This notebook is a narrative walk-through of `analyze_info_capitalization.py`. It loads the constructed four-firm EO panel and reproduces every table and figure in the paper.

**Question.** Satellite-data firms increasingly sell *recurring, contracted* data. Does the market capitalize the depth of that contracted backlog into the provider's own equity multiple?

**Design.** Regress `log(EV/Sales)` on **backlog depth** (RPO / TTM revenue) and revenue growth, under firm and year fixed effects, firm-clustered SEs. Small panel; associations only; robustness reported openly.

In [1]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Load the panel and construct analysis variables

No imputation. Firm-quarters missing any regression component are dropped. Satellogic (SATL) never discloses RPO, so it has no backlog depth and does not enter the regression.

In [2]:
df = pd.read_csv('../data/eo_panel_refined.csv', parse_dates=['end']).sort_values(['ticker','end'])
df['backlog_depth'] = df['rpo_to_rev']
df['growth'] = df['rev_yoy']
df['log_ev_sales'] = np.log(df['ev_sales'])

reg_cols = ['log_ev_sales','backlog_depth','growth']
d = df.dropna(subset=reg_cols)
d = d[np.isfinite(d['log_ev_sales'])].reset_index(drop=True)

print(f'Full panel: {len(df)} firm-quarters, {df.ticker.nunique()} firms')
print(f'Regression sample: {len(d)} firm-quarters, firms = {sorted(d.ticker.unique())}')
print(f'Years: {int(d.year.min())}-{int(d.year.max())}')

Full panel: 61 firm-quarters, 4 firms
Regression sample: 40 firm-quarters, firms = ['BKSY', 'PL', 'SPIR']
Years: 2022-2026


## 2. Table 1 — panel descriptives

In [3]:
labels = {'PL':'Planet Labs','BKSY':'BlackSky','SPIR':'Spire Global','SATL':'Satellogic'}
desc = (d.groupby('ticker')
          .agg(n=('ev_sales','size'),
               ev_sales_mean=('ev_sales','mean'), ev_sales_sd=('ev_sales','std'),
               backlog_depth_mean=('backlog_depth','mean'),
               growth_mean=('growth','mean'))
          .round(2))
desc

,n,ev_sales_mean,ev_sales_sd,backlog_depth_mean,growth_mean
ticker,,,,,
BKSY,13,20.420,15.600,3.080,0.520
PL,13,8.820,10.430,1.310,0.370
SPIR,14,9.780,12.650,2.140,0.360


## 3. Table 2 — nested panel regressions (firm-clustered SEs)

Four specifications: pooled, firm FE, year FE, firm+year FE. Backlog depth is positive throughout; once year effects absorb the 2022–2026 swing in sector multiples it is significant at the 1% level.

In [4]:
def fit(formula):
    return smf.ols(formula, data=d).fit(cov_type='cluster', cov_kwds={'groups': d['ticker']})

specs = {
 'Pooled':       'log_ev_sales ~ backlog_depth + growth',
 'Firm FE':      'log_ev_sales ~ backlog_depth + growth + C(ticker)',
 'Year FE':      'log_ev_sales ~ backlog_depth + growth + C(year)',
 'Firm+Year FE': 'log_ev_sales ~ backlog_depth + growth + C(ticker) + C(year)',
}
rows=[]
for name,f in specs.items():
    m=fit(f)
    rows.append(dict(Model=name, n=int(m.nobs), R2=round(m.rsquared,3),
                     beta_backlog=round(m.params['backlog_depth'],3),
                     p_backlog=round(m.pvalues['backlog_depth'],4),
                     beta_growth=round(m.params['growth'],3),
                     p_growth=round(m.pvalues['growth'],4)))
pd.DataFrame(rows)

,Model,n,R2,beta_backlog,p_backlog,beta_growth,p_growth
0,Pooled,40,0.533,0.461,0.000,1.287,0.000
1,Firm FE,40,0.559,0.491,0.070,1.281,0.000
2,Year FE,40,0.701,0.400,0.000,0.980,0.001
3,Firm+Year FE,40,0.732,0.405,0.000,0.896,0.004


## 4. Table 3 — robustness and the fragility of small-cluster inference

With only three clusters, inference is fragile. We check: (a) HC1 vs clustered SEs, (b) leave-one-firm-out, (c) within-firm demeaning. **The honest finding:** the coefficient stays positive everywhere and survives within-firm demeaning, but loses conventional significance when BlackSky or Planet Labs is dropped. We report this, not around it.

In [5]:
base = 'log_ev_sales ~ backlog_depth + growth + C(ticker) + C(year)'
rob=[]
rob.append(('HC1', smf.ols(base,data=d).fit(cov_type='HC1')))
rob.append(('Cluster by firm', fit(base)))
out=[dict(variant=l, beta=round(m.params['backlog_depth'],3),
          se=round(m.bse['backlog_depth'],3), p=round(m.pvalues['backlog_depth'],4)) for l,m in rob]
for tk in sorted(d.ticker.unique()):
    m=smf.ols(base,data=d[d.ticker!=tk]).fit(cov_type='HC1')
    out.append(dict(variant=f'drop {tk}', beta=round(m.params['backlog_depth'],3),
                    se=round(m.bse['backlog_depth'],3), p=round(m.pvalues['backlog_depth'],4)))
pd.DataFrame(out)

,variant,beta,se,p
0,HC1,0.405,0.195,0.038
1,Cluster by firm,0.405,0.073,0.000
2,drop BKSY,0.267,0.336,0.426
3,drop PL,0.282,0.245,0.251
4,drop SPIR,0.501,0.265,0.059


## 5. Table 4 — univariate associations

In [6]:
for v,lab in [('backlog_depth','Backlog depth'),('growth','Revenue growth')]:
    r,p = stats.pearsonr(d[v], d['log_ev_sales'])
    rho,ps = stats.spearmanr(d[v], d['log_ev_sales'])
    print(f'{lab:16s}  Pearson r={r:.3f} (p={p:.3f})   Spearman rho={rho:.3f} (p={ps:.3f})')

Backlog depth     Pearson r=0.334 (p=0.035)   Spearman rho=0.375 (p=0.017)
Revenue growth    Pearson r=0.587 (p=0.000)   Spearman rho=0.555 (p=0.000)


## 6. Figures

Generated at 600 dpi (PNG + PDF) by `analyze_info_capitalization.py`. Run the script to regenerate:

```bash
python analyze_info_capitalization.py
```

- **Fig 1** backlog depth vs log(EV/Sales), pooled fit
- **Fig 2** backlog coefficient across the four specifications (95% CI)
- **Fig 3** backlog-depth timelines by firm
- **Fig 4** leave-one-firm-out coefficient stability

## 7. Reading

Within this small cohort, deeper contracted data backlog is associated with a richer forward valuation multiple, consistent with the market capitalizing the depth of a recurring data business — an information-capitalization reading. The relationship is not a clean causal estimate: three clusters, a young cohort, and leave-one-out fragility all bound the claim. The contribution is a transparent, fully public first-pass estimate of a question the alternative-data literature has largely left implicit.